# 08C | Crop-group classification and robustness
**Koumbia, Burkina Faso | Ibrahim Kabore**

Notebook 08B showed that identifying six individual crops remains difficult. This notebook tests three predefined groups: cereals (maize and sorghum), legumes (cowpea, groundnut and soybean), and cotton. It compares conventional spectral features with frozen Presto representations, controls class weighting, and evaluates spatial and temporal transfer. The six-crop task remains separate.

Training: 2017, 2018, 2020; development validation: 2021. The 2022 reference observations have already been examined in Notebook 08 for another task; the final 2022 group evaluation here is retrospective, not a fresh independent test. No 2022 outcome may influence candidate selection.

## 1. Setup and global pretrained encoder
Run on Colab with an L4 GPU. The frozen encoder is identical to Notebook 07. Existing six-class classifiers and results remain untouched.

In [ ]:
import sys, subprocess
REVISION = "bf8667902f583ec71fcca6b1a7bd1e48f64000f7"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    f"worldcereal @ git+https://github.com/WorldCereal/worldcereal-classification.git@{REVISION}"])

In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, zipfile, io, requests
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from importlib.metadata import version
from prometheo.models import Presto
from prometheo.models.pooling import PoolingMethods
from prometheo.predictors import Predictors, S1_BANDS, S2_BANDS, METEO_BANDS, DEM_BANDS, NODATAVALUE
from worldcereal.openeo.parameters import DEFAULT_SEASONAL_MODEL_URL
from worldcereal.utils.models import load_model_artifact

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/ESA_EO_Projects")
PROJECT = ROOT / "WorldCereal_Burkina_Faso"
DATA = PROJECT / "data"
OUT = PROJECT / "outputs"
MODELS = PROJECT / "models"
for folder in [OUT, MODELS]:
    folder.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Select a GPU runtime in Colab"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| PromethEO:", version("prometheo"),
      "| WorldCereal:", version("worldcereal"))
assert (DATA / "burkina_extractions_clean.parquet").is_file()
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from scipy.spatial import cKDTree
from pyproj import Transformer
import sklearn, joblib
RESULTS = OUT / "crop_groups"
MODEL_DIR = MODELS / "crop_groups"
for folder in [RESULTS, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
SEED = 42

from sklearn.dummy import DummyClassifier
from sklearn.metrics import log_loss


In [ ]:
EXPECTED_SHA256 = "e16dce35b55b368c1553dd6c6dfe26f39d27818f24a9d710e2922cd4f00fed88"
MODEL_URL = DEFAULT_SEASONAL_MODEL_URL
assert "202608211601.zip" in MODEL_URL, MODEL_URL
french_bundle = ROOT / "WorldCereal_Crop_Mapping/models/2026/global_seasonal_20260821.zip"
bundle = french_bundle if french_bundle.is_file() else MODELS / "global_seasonal_20260821.zip"
if not bundle.is_file():
    r = requests.get(MODEL_URL, timeout=(30,120))
    r.raise_for_status()
    assert zipfile.is_zipfile(io.BytesIO(r.content))
    bundle.write_bytes(r.content)
artifact = load_model_artifact(bundle, cache_root=Path("/content/wc_cache"), encoder_only=True)
config = artifact.manifest
encoder_path = artifact.extract_dir / config["artifacts"]["checkpoints"]["encoder_only"]
fingerprint = hashlib.sha256(encoder_path.read_bytes()).hexdigest()
assert fingerprint == EXPECTED_SHA256, f"Unexpected encoder: {fingerprint}"
declared = (config.get("backbone") or {}).get("fingerprint")
if declared:
    assert fingerprint == declared
encoder = Presto()
encoder.load_state_dict(torch.load(encoder_path, map_location="cpu", weights_only=True), strict=True)
encoder = encoder.to("cuda").eval()
encoder.requires_grad_(False)
print("Verified:", fingerprint)
print("Checkpoint:", bundle)

## 2. Rebuild the matched feature dataset
Read all reference observations and apply the identical quality filter. Feature construction is deterministic and contains no fitted preprocessing; 2022 features are not used in model fitting or selection. Radar is masked because its availability differs substantially between years.

In [ ]:
KEYS = ["ref_id","sample_id"]
META = ["label","label_year","split","spatial_group","lon","lat","ewoc_code"]
BAND_MAP = {**{f"S2-L2A-{b}": b.replace("B0","B") for b in
                ["B02","B03","B04","B05","B06","B07","B08","B8A","B11","B12"]},
    "S1-SIGMA0-VV":"VV", "S1-SIGMA0-VH":"VH",
    "AGERA5-PRECIP":"precipitation", "AGERA5-TMEAN":"temperature",
    "elevation":"elevation", "slope":"slope"}
BANDS = list(BAND_MAP)
FEATURES = [f"presto_ft_{i}" for i in range(128)]
POOL_MONTHS = [5,6,7,8,9,10]
BATCH = 256
raw = pd.read_parquet(DATA / "burkina_extractions_clean.parquet")
assert set(KEYS + META + BANDS + ["timestamp"]) <= set(raw.columns)
assert raw[KEYS + META].notna().all().all()
raw["timestamp"] = pd.to_datetime(raw.timestamp, utc=True).dt.tz_localize(None)
raw["month"] = raw.timestamp.dt.month
raw["label_year"] = raw.label_year.astype(int)
assert raw.timestamp.dt.year.eq(raw.label_year).all()
assert not raw.duplicated(KEYS + ["month"]).any()
assert raw.groupby(KEYS).month.nunique().eq(12).all()
assert raw.groupby(KEYS)[META].nunique().le(1).all().all()
assert np.isfinite(raw[["lon","lat"]]).all().all()
expected_split = {2017:"train",2018:"train",2020:"train",2021:"validation",2022:"test"}
assert raw.split.eq(raw.label_year.map(expected_split)).all()
print("Rows:",len(raw))
print("Samples:", len(raw[KEYS].drop_duplicates()))
display(pd.crosstab(raw.drop_duplicates(KEYS).label,raw.drop_duplicates(KEYS).split,margins=True))

In [ ]:
raw[BANDS] = raw[BANDS].apply(pd.to_numeric, errors="coerce").replace([65535, np.inf, -np.inf],np.nan)
meta = raw.drop_duplicates(KEYS).set_index(KEYS)[META].sort_index()
wide = raw.pivot(index=KEYS,columns="month",values=BANDS).reindex(
    index=meta.index,columns=pd.MultiIndex.from_product([BANDS,range(1,13)]))
values = {b:wide[b].to_numpy(dtype=np.float32) for b in BANDS}
display(pd.DataFrame({b:pd.Series(a.ravel()).describe() for b,a in values.items()}).T)
optical = np.isfinite(values["S2-L2A-B08"])
radar = np.isfinite(values["S1-SIGMA0-VV"]) & (values["S1-SIGMA0-VV"]>0)
keep = (optical.sum(1)>=6) & (optical[:,4:10].sum(1)>=3)
print("Before common quality filter:",len(meta), "| retained:",int(keep.sum()))
meta = meta.iloc[np.flatnonzero(keep)].reset_index()
values = {b:a[keep] for b,a in values.items()}
meta["optical_months"] = optical[keep].sum(1)
meta["radar_months"] = radar[keep].sum(1)
display(pd.crosstab(meta.label,meta.split,margins=True))
assert set(meta.label)=={"Maize","Cotton","Sorghum","Soybean","Cowpea","Groundnut"}
assert (pd.crosstab(meta.label,meta.split)>0).all().all()
display(meta.groupby("label_year")[["optical_months","radar_months"]].mean().round(2))
assert set(meta.split)=={"train","validation","test"}
train_idx = np.flatnonzero(meta.split.eq("train"))
val_idx = np.flatnonzero(meta.split.eq("validation"))
Y = meta.label.to_numpy()
CLASSES = sorted(set(Y))
assert len(train_idx)==992 and len(val_idx)==271

test_idx = np.flatnonzero(meta.split.eq("test"))
assert len(test_idx)==262


In [ ]:
def ratio(a,b):
    return np.divide(a-b,a+b,out=np.full_like(a,np.nan),where=np.isfinite(a+b)&((a+b)!=0))
ndvi = ratio(values["S2-L2A-B08"],values["S2-L2A-B04"])


In [ ]:
def encode(values, lon, lat, dates, use_radar):
    n,t = len(lon),len(dates)
    assert t==12
    s1 = np.full((n,1,1,t,len(S1_BANDS)),NODATAVALUE,np.float32)
    s2 = np.full((n,1,1,t,len(S2_BANDS)),NODATAVALUE,np.float32)
    meteo = np.full((n,1,1,t,len(METEO_BANDS)),NODATAVALUE,np.float32)
    dem = np.full((n,1,1,len(DEM_BANDS)),NODATAVALUE,np.float32)

    for source,band in BAND_MAP.items():
        a = np.asarray(values[source],dtype=np.float32).copy()
        assert a.shape==(n,t)
        ok = np.isfinite(a) & (a!=NODATAVALUE) & (a!=65535)
        a[~ok] = NODATAVALUE
        if band in S2_BANDS:
            if band=="B8A":
                a[:] = NODATAVALUE
            s2[:,0,0,:,S2_BANDS.index(band)] = a
        elif band in S1_BANDS:
            if use_radar:
                ok &= a>0
                a[~ok] = NODATAVALUE
                a[ok] = 20*np.log10(a[ok])-83
                s1[:,0,0,:,S1_BANDS.index(band)] = a
        elif band in METEO_BANDS:
            a[ok] /= 100000 if band=="precipitation" else 100
            meteo[:,0,0,:,METEO_BANDS.index(band)] = a
        elif band in DEM_BANDS:
            dem[:,0,0,DEM_BANDS.index(band)] = a[:,0]
        else:
            raise ValueError(band)

    dates = pd.DatetimeIndex(dates)
    stamps = np.stack([dates.day,dates.month,dates.year],axis=1)
    predictor = Predictors(s1=s1,s2=s2,meteo=meteo,dem=dem,
        latlon=np.stack([lat,lon],axis=1).astype("float32").reshape(n,1,1,2),
        timestamps=np.broadcast_to(stamps,(n,t,3)).copy())
    with torch.inference_mode():
        z = encoder(predictor,eval_pooling=PoolingMethods.TIME).reshape(n,t,128)
        keep = np.flatnonzero(dates.month.isin(POOL_MONTHS)).tolist()
        return z[:,keep,:].cpu().numpy()

In [ ]:
def run_batch(ix, use_radar):
    yrs = meta.label_year.to_numpy()[ix]
    assert len(set(yrs))==1
    dates = pd.date_range(f"{yrs[0]}-01-01",periods=12,freq="MS")
    return encode({b:a[ix] for b,a in values.items()},
                  meta.lon.to_numpy()[ix],meta.lat.to_numpy()[ix],dates,use_radar)

Z = np.empty((len(meta),6,128),dtype=np.float32)
for year in sorted(meta.label_year.unique()):
    ids = np.flatnonzero(meta.label_year.eq(year))
    for start in tqdm(range(0,len(ids),BATCH),desc=str(year)):
        ix = ids[start:start+BATCH]
        Z[ix] = run_batch(ix,False)
assert np.isfinite(Z).all()
print("Monthly vectors:",Z.shape)
seasonal = Z.mean(axis=1)
old = pd.read_parquet(OUT/"reference_embeddings_optical.parquet")
old = old.set_index(KEYS).loc[pd.MultiIndex.from_frame(meta[KEYS])]
assert np.array_equal(old.label.to_numpy(),Y)
np.testing.assert_allclose(seasonal,old[FEATURES].to_numpy(),rtol=1e-4,atol=1e-5)
print("Seasonal vectors agree with Notebook 07.")


In [ ]:
X = {"seasonal":seasonal,"stages":Z.reshape(-1,3,2,128).mean(axis=2).reshape(-1,384),
     "monthly":Z.reshape(-1,768)}
names = {"seasonal":FEATURES,
         "stages":[f"{s}_ft_{i}" for s in ["may_jun","jul_aug","sep_oct"] for i in range(128)],
         "monthly":[f"m{m:02d}_ft_{i}" for m in POOL_MONTHS for i in range(128)]}
spectral = {b:values[b]/10000 for b in BANDS if b.startswith("S2") and not b.endswith("B8A")}
spectral.update({"NDVI":ndvi,"NDMI":ratio(values["S2-L2A-B08"],values["S2-L2A-B11"]),
                 "NDRE":ratio(values["S2-L2A-B08"],values["S2-L2A-B05"]),
                 "precipitation":values["AGERA5-PRECIP"]/100000,
                 "temperature":values["AGERA5-TMEAN"]/100})
X["spectral"] = np.column_stack([a[:,4:10] for a in spectral.values()]+[values[b][:,0] for b in ["elevation","slope"]])
names["spectral"] = [f"{b}_m{m:02d}" for b in spectral for m in POOL_MONTHS]+["elevation","slope"]
for name,arr in X.items():
    assert arr.shape==(len(meta),len(names[name]))
    print(name,arr.shape)


## 3. Define crop groups
These groups are a deliberately coarser classification scheme, not a complete land-cover legend. Millet, rice, fallow and other omitted classes cannot be identified by this classifier.

In [ ]:
GROUP_MAP = {"Maize":"Cereals","Sorghum":"Cereals","Cowpea":"Legumes","Groundnut":"Legumes","Soybean":"Legumes","Cotton":"Cotton"}
Y6 = Y.copy()
Y = np.array([GROUP_MAP[y] for y in Y6])
CLASSES = sorted(set(Y))
display(pd.crosstab(Y,meta.split,margins=True))
assert CLASSES==["Cereals","Cotton","Legumes"]


## 4. Compare models on 2021 validation
We retain the previous RF and HGB settings, add balanced HGB, and retain balanced regularised logistic regression. Imputation and scaling are fitted only on the training partition. A majority-class baseline provides context. The selection rule is highest validation macro F1, then balanced accuracy, then deterministic name ordering.

In [ ]:
def candidate(name,model):
    estimators = {
        "rf_balanced":RandomForestClassifier(n_estimators=150,min_samples_leaf=2,class_weight="balanced_subsample",n_jobs=2,random_state=SEED),
        "hgb":HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=31,learning_rate=.1,random_state=SEED),
        "hgb_balanced":HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=31,learning_rate=.1,class_weight="balanced",random_state=SEED),
        "lr_balanced":LogisticRegression(C=1,class_weight="balanced",max_iter=2500,random_state=SEED)
    }
    steps = []
    if name=="spectral":
        steps.append(SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True))
    if model=="lr_balanced":
        steps.append(StandardScaler())
    return make_pipeline(*steps,estimators[model])

def metrics(y,p):
    return {"accuracy":accuracy_score(y,p),
            "balanced_accuracy":balanced_accuracy_score(y,p),
            "macro_f1":f1_score(y,p,labels=CLASSES,average="macro",zero_division=0)}
MODELNAMES = ["rf_balanced","hgb","hgb_balanced","lr_balanced"]
rows, fitted = [], {}
for name,arr in X.items():
    for model in MODELNAMES:
        pipe = candidate(name,model).fit(arr[train_idx],Y[train_idx])
        fitted[name,model] = pipe
        rows.append({"configuration":name,"model":model,**metrics(Y[val_idx],pipe.predict(arr[val_idx]))})
dummy = DummyClassifier(strategy="most_frequent").fit(X["seasonal"][train_idx],Y[train_idx])
baseline = metrics(Y[val_idx],dummy.predict(X["seasonal"][val_idx]))
validation = pd.DataFrame(rows).sort_values(["macro_f1","balanced_accuracy","configuration","model"],ascending=[False,False,True,True])
display(validation.round(4))
print("Majority baseline:",baseline)
validation.to_csv(RESULTS/"validation_metrics.csv",index=False)
winner = validation.iloc[0]
name_win, model_win = winner.configuration,winner.model
selected = fitted[name_win,model_win]
decision = {"configuration":name_win,"model":model_win,"validation_macro_f1":float(winner.macro_f1),"selection_metric":"2021_validation_macro_f1"}
(RESULTS/"model_selection.json").write_text(json.dumps(decision,indent=2))
ax = validation.pivot(index="configuration",columns="model",values="macro_f1").plot.bar(figsize=(11,4),rot=0)
ax.axhline(baseline["macro_f1"],ls="--",color="black",label="Majority baseline")
ax.set(title="2021 | Three crop groups",ylabel="Macro F1",ylim=(0,1))
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS/"validation_comparison.png",dpi=180)
plt.show()
print("Frozen selection:",decision)


## 5. Spatial and temporal cross-validation
Use the same six-crop stratification and spatial grouping as Notebook 08B to maintain comparable fold assignments. The three-group outcome is used for training and scoring. Exclude training points within 100 m of held-out points in spatial folds. Leave-year-out folds test historical temporal transfer, not unseen locations. These diagnostics do not alter the frozen selection.

In [ ]:
groups = meta.spatial_group.to_numpy()
xx,yy = Transformer.from_crs(4326,32630,always_xy=True).transform(meta.lon.to_numpy(),meta.lat.to_numpy())
xy = np.column_stack([xx,yy])
folds = []
cv = StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
for fold,(tr,te) in enumerate(cv.split(X["seasonal"][train_idx],Y6[train_idx],groups[train_idx]),1):
    ti,hi = train_idx[tr],train_idx[te]
    assert set(groups[ti]).isdisjoint(groups[hi])
    dist,_ = cKDTree(xy[hi]).query(xy[ti])
    folds.append(("spatial_100m",str(fold),ti[dist>100],hi))
for year in sorted(meta.label_year.iloc[train_idx].unique()):
    hi = train_idx[meta.label_year.iloc[train_idx].to_numpy()==year]
    ti = np.setdiff1d(train_idx,hi)
    folds.append(("leave_year_out",str(year),ti,hi))
cv_rows, audit = [], []
for scheme,fold,ti,hi in tqdm(folds):
    complete = set(Y[ti])==set(CLASSES) and set(Y[hi])==set(CLASSES)
    audit.append({"scheme":scheme,"fold":fold,"train":len(ti),"held_out":len(hi),"complete":complete})
    if not complete:
        print("Skipped incomplete fold:",scheme,fold)
        continue
    for name,arr in X.items():
        for model in MODELNAMES:
            pipe = candidate(name,model).fit(arr[ti],Y[ti])
            cv_rows.append({"scheme":scheme,"fold":fold,"configuration":name,"model":model,
                            **metrics(Y[hi],pipe.predict(arr[hi]))})
pd.DataFrame(audit).to_csv(RESULTS/"fold_audit.csv",index=False)
cv_scores = pd.DataFrame(cv_rows)
cv_scores.to_csv(RESULTS/"cv_metrics.csv",index=False)
summary = cv_scores.groupby(["scheme","configuration","model"])[["accuracy","balanced_accuracy","macro_f1"]].agg(["mean","std","count"])
display(summary.round(4))
summary.to_csv(RESULTS/"cv_summary.csv")

print("Frozen candidate cross-validation:")
display(summary.loc[(slice(None),name_win,model_win),:].round(4))


## 6. Selected model diagnostics and retrospective 2022 assessment
The model was selected using 2021 only. Evaluate the single frozen model on 2022, without changing any settings afterward. Report per-class metrics, confusion matrices, log loss and confidence reliability. Maximum class probability is an uncalibrated model score, not a guaranteed probability of correctness. Reliability bins describe performance; they do not calibrate the model.

In [ ]:
assert list(selected.classes_)==CLASSES
report_rows = []
prediction_frames = []
for split,idx in [("validation_2021",val_idx),("retrospective_2022",test_idx)]:
    probs = selected.predict_proba(X[name_win][idx])
    pred = selected.predict(X[name_win][idx])
    conf = probs.max(axis=1)
    correct = pred==Y[idx]
    result = {**metrics(Y[idx],pred),"log_loss":log_loss(Y[idx],probs,labels=CLASSES)}
    report_rows.append({"split":split,**result})
    print(split,result)
    report = pd.DataFrame(classification_report(Y[idx],pred,labels=CLASSES,output_dict=True,zero_division=0)).T
    display(report.round(3))
    report.to_csv(RESULTS/f"{split}_report.csv")
    fig,ax = plt.subplots(figsize=(6,5))
    ConfusionMatrixDisplay.from_predictions(Y[idx],pred,labels=CLASSES,normalize="true",cmap="Blues",values_format=".2f",ax=ax)
    ax.grid(False)
    ax.set_title(split)
    fig.tight_layout()
    fig.savefig(RESULTS/f"{split}_confusion.png",dpi=180)
    plt.show()
    frame=meta.iloc[idx][KEYS+["label","label_year","split"]].copy()
    frame["group_true"]=Y[idx]
    frame["group_pred"]=pred
    frame["confidence"]=conf
    frame["correct"]=correct
    for j,label in enumerate(CLASSES):
        frame[f"prob_{label}"]=probs[:,j]
    prediction_frames.append(frame)
    bins=pd.cut(conf,np.linspace(0,1,11),include_lowest=True)
    rel=pd.DataFrame({"confidence":conf,"correct":correct,"bin":bins}).groupby("bin",observed=True).agg(n=("correct","size"),mean_confidence=("confidence","mean"),accuracy=("correct","mean")).reset_index()
    display(rel.round(3))
    rel.to_csv(RESULTS/f"{split}_reliability.csv",index=False)
    fig,ax=plt.subplots(figsize=(5,4))
    ax.plot([0,1],[0,1],"--",color="gray")
    ax.scatter(rel.mean_confidence,rel.accuracy,s=20+rel.n*2)
    ax.set(xlim=(0,1),ylim=(0,1),xlabel="Mean predicted confidence",ylabel="Observed accuracy",title=split)
    fig.tight_layout()
    fig.savefig(RESULTS/f"{split}_reliability.png",dpi=180)
    plt.show()
pd.DataFrame(report_rows).to_csv(RESULTS/"selected_metrics.csv",index=False)
pd.concat(prediction_frames,ignore_index=True).to_csv(RESULTS/"selected_predictions.csv",index=False)


## 7. Export the dedicated group classifier
Save the training-only model and its exact feature contract for Notebook 09. The prediction map must use the same feature order, transformations, calendar and missing-data rules. A separate cropland mask is required; these three classes cannot represent all crops. The 2022 retrospective score is not a guarantee of accuracy for a later mapping year.

In [ ]:
path = MODEL_DIR/"selected_group_classifier.joblib"
joblib.dump(selected,path)
manifest = {**decision,"feature_names":names[name_win],"feature_count":len(names[name_win]),
            "checkpoint_sha256":fingerprint,"worldcereal_revision":REVISION,
            "worldcereal":version("worldcereal"),"prometheo":version("prometheo"),
            "sklearn_version":sklearn.__version__,"context":"Jan-Dec","pool_months":POOL_MONTHS,
            "use_radar":False,"mask_b8a":True,"fit_years":[2017,2018,2020],
            "validation_year":2021,"retrospective_year":2022,
            "retrospective_note":"2022 already examined for six-crop task; not used to select this classifier",
            "classifier_file":path.name,"classifier_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
            "classes":CLASSES,"crop_codes":{str(i+1):v for i,v in enumerate(CLASSES)},
            "group_mapping":GROUP_MAP,"quality_filter":"B08 >=6 annual months and >=3 May-Oct months",
            "feature_contract":"Notebook 08C section 2, identical to Notebook 08B section 5",
            "confidence":"uncalibrated maximum predicted probability"}
(MODEL_DIR/"classifier_manifest.json").write_text(json.dumps(manifest,indent=2))
print("Saved:",path)
print("Selected:",decision)
print("Crop codes:",manifest["crop_codes"])
display(pd.DataFrame(report_rows).round(4))
print("Results:",RESULTS)
